In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

df = pd.read_csv('DATASET/attrition.csv')

print(df.shape)
print(df.head())
print(df.info())
print(df['Attrition'].value_counts())

(1470, 35)
   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1                 8          1  Life Sciences              1               2   
2                 2          2          Other              1               4   
3                 3          4  Life Sciences              1               5   
4                 2          1        Medical              1               7   

   ...  RelationshipSatisfaction StandardHours  StockOptionLeve

In [2]:
# Drop constant columns that add zero information
df.drop(columns=['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber'], inplace=True)

# Encode target variable
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Encode binary string columns
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['OverTime'] = df['OverTime'].map({'Yes': 1, 'No': 0})

print(df.shape)
print(df['Attrition'].value_counts())

(1470, 31)
Attrition
0    1233
1     237
Name: count, dtype: int64


In [3]:
# Remaining categorical columns
cat_cols = ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus']

df = pd.get_dummies(df, columns=cat_cols, drop_first=False)

print(df.shape)
print(df.dtypes.value_counts())

(1470, 50)
int64    26
bool     24
Name: count, dtype: int64


In [4]:
# Convert all bool columns to int
df[df.select_dtypes(include='bool').columns] = df.select_dtypes(include='bool').astype(int)

print(df.dtypes.value_counts())
print(df.shape)

int64    50
Name: count, dtype: int64
(1470, 50)


In [5]:
# Compensation ratio — income relative to job level
df['CompensationRatio'] = df['MonthlyIncome'] / (df['JobLevel'] + 1)

# Tenure per job — how long per company worked at
df['TenurePerJob'] = df['YearsAtCompany'] / (df['NumCompaniesWorked'] + 1)

# Years without change — stagnation indicator
df['YearsWithoutChange'] = df['YearsInCurrentRole'] + df['YearsSinceLastPromotion']

print(df.shape)
print(df[['CompensationRatio', 'TenurePerJob', 'YearsWithoutChange']].head())

(1470, 53)
   CompensationRatio  TenurePerJob  YearsWithoutChange
0        1997.666667      0.666667                   4
1        1710.000000      5.000000                   8
2        1045.000000      0.000000                   0
3        1454.500000      4.000000                  10
4        1734.000000      0.200000                   4


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['Attrition'])
y = df['Attrition']

# Split first — always before SMOTE and scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale after split — fit only on train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train shape: {X_train_scaled.shape}")
print(f"Test shape: {X_test_scaled.shape}")
print(f"Train class distribution:\n{y_train.value_counts()}")
print(f"Test class distribution:\n{y_test.value_counts()}")

Train shape: (1176, 52)
Test shape: (294, 52)
Train class distribution:
Attrition
0    986
1    190
Name: count, dtype: int64
Test class distribution:
Attrition
0    247
1     47
Name: count, dtype: int64


In [7]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE train shape: {X_train_res.shape}")
print(f"After SMOTE class distribution:\n{pd.Series(y_train_res).value_counts()}")

After SMOTE train shape: (1972, 52)
After SMOTE class distribution:
Attrition
0    986
1    986
Name: count, dtype: int64


In [8]:
import pickle

# Save scaled arrays and target
pickle.dump((X_train_res, X_test_scaled, y_train_res, y_test), open('../artifacts/scaler.pkl', 'wb'))

# Save scaler for later use in deployment
pickle.dump(scaler, open('../artifacts/scaler.pkl', 'wb'))

# Save feature names for SHAP later
feature_names = X.columns.tolist()
pickle.dump(feature_names, open('../artifacts/scaler.pkl', 'wb'))

print("Saved: preprocessed_data.pkl, scaler.pkl, feature_names.pkl")
print(f"Features: {len(feature_names)}")

Saved: preprocessed_data.pkl, scaler.pkl, feature_names.pkl
Features: 52


In [9]:
os.makedirs('../artifacts', exist_ok=True)

pickle.dump((X_train_res, X_test_scaled, y_train_res, y_test), open('../artifacts/preprocessed_data.pkl', 'wb'))
pickle.dump(scaler, open('../artifacts/scaler.pkl', 'wb'))
pickle.dump(feature_names, open('../artifacts/feature_names.pkl', 'wb'))

print("Saved to artifacts/")

Saved to artifacts/
